# 03 — Knowledge Graph & Search Pipeline

This notebook demonstrates:
1. Building the audience knowledge graph (NetworkX)
2. Exploring graph structure and relationships
3. Running the full search pipeline
4. Evaluation against test queries

In [ ]:
import sys
sys.path.insert(0, '..')

from collections import defaultdict
import networkx as nx

from data_loader import load_all
from embedder import load_artifacts, load_model
from clustering import load_clusters
from graph_builder import build_graph, load_graph
from query import create_engine, format_result
from evaluate import run_evaluation, TEST_CASES

## Step 1: Build or Load the Graph

In [ ]:
# Load pre-built graph
segments = load_all()
embeddings, index, segment_ids = load_artifacts(prefix='raw')
groups, labels, centroids = load_clusters(prefix='raw')
graph = load_graph()

print(f"Nodes: {graph.number_of_nodes()}")
print(f"Edges: {graph.number_of_edges()}")

# Edge type distribution
edge_types = defaultdict(int)
for u, v, d in graph.edges(data=True):
    edge_types[d.get('edge_type', 'unknown')] += 1
print("\nEdge types:")
for et, count in sorted(edge_types.items()):
    print(f"  {et}: {count}")

## Step 2: Explore Graph Neighborhoods

Pick a segment and explore its relationships.

In [ ]:
# Find a segment and explore its graph neighborhood
def explore_node(G, node_id, depth=1):
    """Show all edges from a node."""
    if node_id not in G:
        print(f"Node {node_id} not in graph")
        return
    
    node_data = G.nodes[node_id]
    print(f"Node: {node_id}")
    for k, v in node_data.items():
        print(f"  {k}: {v}")
    
    print(f"\nOutgoing edges ({G.out_degree(node_id)}):")
    for _, target, data in G.edges(node_id, data=True):
        target_name = G.nodes[target].get('name', target)
        print(f"  --[{data.get('edge_type')}]--> {target_name} (weight: {data.get('weight', 'N/A')})")
    
    print(f"\nIncoming edges ({G.in_degree(node_id)}):")
    for source, _, data in G.in_edges(node_id, data=True):
        source_name = G.nodes[source].get('name', source)
        print(f"  <--[{data.get('edge_type')}]-- {source_name}")

# Explore a group node
group_nodes = [n for n, d in graph.nodes(data=True) if d.get('node_type') == 'group']
print(f"Total group nodes: {len(group_nodes)}")
explore_node(graph, group_nodes[0])

In [ ]:
# Find EQUIVALENT_TO edges — these are the cross-platform matches
equiv_edges = [(u, v, d) for u, v, d in graph.edges(data=True) if d.get('edge_type') == 'EQUIVALENT_TO']
print(f"EQUIVALENT_TO edges: {len(equiv_edges)}")

# Show some examples
print("\nCross-platform equivalent pairs:")
for u, v, d in equiv_edges[:15]:
    u_data = graph.nodes[u]
    v_data = graph.nodes[v]
    print(f"  [{u_data.get('platform', '?'):10s}] {u_data.get('name', u):35s} <=> [{v_data.get('platform', '?'):10s}] {v_data.get('name', v):35s} (sim: {d.get('weight', 0):.3f})")

## Step 3: Full Search Pipeline

In [ ]:
engine = create_engine()

# Run a search
result = engine.search("luxury SUV shoppers")
print(format_result(result))

In [ ]:
# Try more queries
queries = [
    "parents shopping for baby products",
    "tech-savvy millennials",
    "outdoor adventure enthusiasts",
]

for q in queries:
    result = engine.search(q)
    print(format_result(result))

## Step 4: Evaluation

In [ ]:
results = run_evaluation(engine)

In [ ]:
# Visualize evaluation results
import plotly.graph_objects as go

queries = [r['query'] for r in results]
keyword_recalls = [r['keyword_recall'] * 100 for r in results]
platform_coverages = [r['platform_coverage'] * 100 for r in results]

fig = go.Figure(data=[
    go.Bar(name='Keyword Recall', x=queries, y=keyword_recalls, marker_color='#FF6B6B'),
    go.Bar(name='Platform Coverage', x=queries, y=platform_coverages, marker_color='#4ECDC4'),
])
fig.update_layout(
    title='Evaluation: Keyword Recall & Platform Coverage per Query',
    template='plotly_dark',
    barmode='group',
    yaxis_title='Percentage (%)',
    xaxis_tickangle=-45,
    width=1200, height=500,
)
fig.show()